# 임상 ML 학습

`model_choice.yaml`에서 모델 목록을, 각 모델 `parameter.yaml`에서 하이퍼파라미터·early stopping·threshold를 설정합니다. 전처리 기본값은 코드에서 관리합니다. `model_version`으로 입력 변수 묶음(1~4)을 고릅니다. 학습은 환자 ID로 고정 split을 적용하고 train에만 전처리를 fit합니다. test는 자동 평가하지 않습니다.

- 저장 모델을 불러오거나 YAML 고정값으로 학습하려면 **학습 실행 → 결과 확인**을 실행합니다. `model_path`가 있으면 재학습 없이 기존 run을 사용하고, 없으면 YAML 파라미터로 학습합니다.
- Optuna만 사용하려면 아래 **공통 import 셀과 입력 변수 버전 선택 셀**을 실행한 뒤 **Optuna 실험**으로 이동합니다. 일반 학습·결과 확인 셀은 건너뛰어도 됩니다.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Members 패키지를 찾도록 저장소 루트를 Python 검색 경로에 등록합니다.
repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/ML/model_choice.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("sixletters-poc 저장소 내부에서 노트북을 실행하세요")
    repo = repo.parent

sys.path.insert(0, str(repo))
choice_path = repo / "Members/sang/model/ML/model_choice.yaml"

from Members.sang.model.ML.training import train_clinical  # noqa: E402

print("Members import 준비 완료:", repo)

## 입력 변수 버전 선택

`model_version`을 **1, 2, 3, 4 중 하나**로 지정합니다. 기본값 4는 기존 전체 23개 입력을 유지합니다. 모든 알고리즘(XGBoost·CatBoost·Random Forest)과 Optuna에 같은 버전을 적용합니다.

| model_version | 입력 변수 수 | 구성 |
|---|---:|---|
| 1 | 4 | 성별, 나이, LVEF, NYHA |
| 2 | 8 | 버전 1 + eGFR, 허혈성 원인, Albumin, 당뇨 |
| 3 | 21 | 버전 2 + 제공한 심장 구조·기능, 병력, 혈압, Pro-BNP, 심흉비 |
| 4 | 23 | 버전 3 + 흡연량, 음주량 |

정확한 열 목록과 순서는 `ML/feature_versions.py`에 있습니다. `Mitral valve insufficiency `의 끝 공백은 CSV 로더와 동일하게 정리했습니다. 범주형 열도 선택한 입력 변수 안에서만 처리합니다. 아래 출력에서 실제 입력 열을 확인하세요.

환자 split은 모든 버전에서 같습니다. 버전이 다른 저장 모델은 재사용하지 않으며, 저장 결과에 알고리즘 이름과 별도로 `model_version`·`feature_names`를 기록합니다. 버전 비교와 최적값 선택은 validation으로 하고 test는 최종 평가에 사용합니다.

In [ ]:
from Members.sang.model.ML.feature_versions import get_feature_columns

model_version = 4  # 1: 4개, 2: 8개, 3: 21개, 4: 23개
input_columns = get_feature_columns(model_version)

print("model_version:", model_version)
print("입력 변수 수:", len(input_columns))
input_columns

## 학습 실행

선택된 각 모델의 `parameter.yaml`에서 `model_path`를 먼저 확인합니다. 선택한 `model_version`과 변수 목록이 일치하는 파일이 있으면 재사용하고, 해당 버전의 파일이 없으면 YAML의 `model`·`training`·`classification` 값으로 학습합니다. 새로 학습한 모델의 경로는 해당 YAML의 `model_path`에 자동 기록합니다.

`model_path`는 이 저장소가 저장한 `best_model.json`·`best_model.cbm`·`best_model.joblib` 파일 또는 run 디렉터리입니다. 파일 옆의 metadata·설정·validation 결과와 필요한 전처리 파일도 함께 있어야 합니다. 파일이 존재하지만 run 구성이 불완전하면 오류를 표시합니다.

XGBoost·CatBoost는 validation AP로 best iteration을 선택합니다. Random Forest는 설정한 tree 수로 학습하고 validation AP를 기록합니다. 저장 모델이 있어도 새로 학습하려면 `train_clinical(choice_path, model_version=model_version, reuse_saved=False)`를 호출합니다. Optuna 셀은 이 옵션으로 항상 새 trial을 학습합니다.

In [ ]:
run_dirs = train_clinical(choice_path, model_version=model_version)
run_dirs

## 결과 확인

각 run은 `ML/models/<run_id>/`에 저장됩니다. 여기에는 best model, train-fitted preprocessing, resolved YAML, audit, validation 환자 확률, 지표가 있습니다. Test 평가는 `evaluation/evaluation.py`에서 별도로 실행합니다.

In [ ]:
for run_dir in run_dirs:
    print(run_dir.name)
    display(pd.read_csv(run_dir / "validation_predictions.csv").head())
    print(json.loads((run_dir / "metrics.json").read_text())["validation"])

## Optuna 실험 (선택)

공통 import 셀과 입력 변수 버전 선택 셀을 먼저 실행합니다. `model_name`은 알고리즘, `model_version`은 입력 변수 묶음입니다. 버전을 바꿀 때마다 Optuna 준비 셀부터 다시 실행해 별도 study로 비교합니다. `model_name`에서 모델 하나를 선택하고 `n_trials`에 실험 횟수를 적습니다. 확장된 탐색의 기본값은 100회입니다. 시간이 오래 걸리면 이 숫자를 줄여 실행 시간을 먼저 확인하세요. 모델을 바꾸려면 이 준비 셀부터 다시 실행합니다.

Optuna는 `Global`의 `ml` extra에 포함되어 있습니다. 현재 커널에서 import가 안 되면 저장소 루트의 터미널에서 `uv sync --project Global --locked --extra ml --inexact`를 실행하고 커널을 재시작합니다. 프로젝트가 요구하는 uv 버전은 `0.9.26`입니다.

환자 split과 YAML의 threshold·조기 종료 횟수는 그대로 사용합니다. trial 중에는 원본 YAML을 수정하지 않고 모델 파라미터만 전달합니다. 탐색 실행 셀이 성공적으로 끝나면 최적 모델만 보관하고 YAML의 `model_path`·버전별 `model_paths`에 경로를 기록합니다. 기존 `model_path`가 있어도 Optuna 실험은 새로 수행합니다.

`mlflow_logging=True`이면 탐색이 성공한 뒤 최적 모델 하나와 파라미터·validation 지표를 MLflow로 전송합니다. 이 PC의 서버 주소는 `http://127.0.0.1:5000`이고 접속 정보는 Git에서 제외되는 `.poc.local.toml`에서 읽습니다. 비밀번호를 노트북에 적지 않습니다. 로컬 실험만 하려면 `False`로 바꿉니다. 서버의 run 한 개는 완료된 study의 best 결과를 뜻하며 trial별 모델을 업로드하지 않습니다.

In [ ]:
import optuna
from Members.sang.model.ML.optuna_experiment import ClinicalOptunaExperiment

model_name = "XGBoost"  # "XGBoost", "CatBoost", "Random Forest" 중 하나
n_trials = 100
mlflow_logging = True  # 탐색 완료 후 best 모델·파라미터·validation 지표 전송

if model_name not in ["XGBoost", "CatBoost", "Random Forest"]:
    raise ValueError("model_name에 지원되는 모델 이름을 입력하세요")
if n_trials < 1:
    raise ValueError("n_trials는 1 이상이어야 합니다")

experiment = ClinicalOptunaExperiment(
    choice_path, model_name, model_version, mlflow_logging=mlflow_logging
)
parameter_path = experiment.parameter_path
print("선택한 모델 설정:", parameter_path)
print("최적 모델 저장 폴더:", experiment.best_directory)

## 탐색 범위와 학습 함수

아래 숫자와 목록이 모델별 탐색 범위입니다. `suggest_int`는 정수, `suggest_float`는 실수, `suggest_categorical`은 목록에서 선택합니다. `log=True`는 작은 값과 큰 값의 규모를 함께 탐색할 때 사용합니다. 자세한 범위와 역할은 `ML/PARAMETER_GUIDE.md`의 **Optuna 확장 탐색 범위**를 참고하세요.

- XGBoost: 트리 수·깊이·학습률 + 자식 노드 조건·분할 규제·행/열 샘플링·L1/L2 규제·양성 가중치
- CatBoost: 반복 수·깊이·학습률 + L2 규제·분할 무작위성·열 샘플링·수치 분할 개수·클래스 가중치·bootstrap
- Random Forest: 트리 수 + 깊이·노드/leaf 최소 샘플·feature 샘플링·분할 기준·클래스 가중치·가지치기·bootstrap

CatBoost는 Bayesian일 때 `bagging_temperature`, Bernoulli일 때 `subsample`만 탐색합니다. Random Forest는 bootstrap이 켜져 있을 때만 `max_samples`를 탐색합니다. 이 조건을 유지해야 호환되지 않는 조합을 피할 수 있습니다. CatBoost `auto_class_weights`와 별도의 `class_weights`·`scale_pos_weight`는 함께 설정하지 마세요.

매 trial에서 **train으로 학습 → validation AP 계산**을 수행합니다. XGBoost·CatBoost의 조기 종료 횟수는 YAML 값을 사용합니다. Random Forest는 설정한 tree 수를 학습합니다. Threshold는 이번 탐색 대상이 아니며 test를 파라미터 선택에 사용하지 않습니다.

범위를 넓힌다고 성능 개선이 보장되지는 않습니다. `n_trials`는 탐색할 조합 수이므로 변수 개수가 늘면 같은 10회로는 일부 조합만 평가합니다. 더 탐색하려면 준비 셀의 `n_trials`를 늘리세요. 기본 TPE는 첫 10회에 무작위 탐색을 사용합니다. 작은 validation 결과에 맞춰 지나치게 조정하지 말고, 모델·버전·threshold를 확정한 뒤 최종 test를 평가합니다.

In [ ]:
def objective(trial):
    if model_name == "XGBoost":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1500, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0, step=0.5),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
            "reg_alpha": trial.suggest_categorical("reg_alpha", [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),
            "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, 2.0, 4.0, 8.0]),
        }
    elif model_name == "CatBoost":
        params = {
            "iterations": trial.suggest_int("iterations", 100, 1500, step=100),
            "depth": trial.suggest_int("depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 30.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.0, 5.0, step=0.5),
            "rsm": trial.suggest_float("rsm", 0.6, 1.0, step=0.1),
            "border_count": trial.suggest_categorical("border_count", [16, 32, 64, 128]),
            "auto_class_weights": trial.suggest_categorical(
                "auto_class_weights", ["None", "Balanced", "SqrtBalanced"]
            ),
            "bootstrap_type": trial.suggest_categorical(
                "bootstrap_type", ["Bayesian", "Bernoulli"]
            ),
        }
        if params["bootstrap_type"] == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 5.0, step=0.5
            )
        else:
            params["subsample"] = trial.suggest_float("subsample", 0.6, 1.0, step=0.1)
    else:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
            "max_depth": trial.suggest_categorical("max_depth", [None, 2, 3, 4, 5, 6, 8, 10, 15]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 12),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, None]),
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "class_weight": trial.suggest_categorical(
                "class_weight", [None, "balanced", "balanced_subsample"]
            ),
            "ccp_alpha": trial.suggest_categorical("ccp_alpha", [0.0, 0.0001, 0.001, 0.005, 0.01]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }
        if params["bootstrap"]:
            params["max_samples"] = trial.suggest_float("max_samples", 0.6, 1.0, step=0.1)
        else:
            params["max_samples"] = None

    return experiment.run_trial(trial, params)

## 탐색 실행

validation AP가 클수록 좋은 trial을 선택합니다. 모델 학습 seed와 Optuna 제안 seed는 기본 42입니다.

**trial 모델은 임시 폴더에서 학습하고 바로 정리합니다.** 탐색 중에는 현재 best 후보 하나만 유지합니다. 탐색이 성공적으로 끝나면 해당 알고리즘·버전의 최적 모델 한 개를 `ML/models/best/<알고리즘>_v<버전>/`에 저장합니다. 예: `ML/models/best/xgboost_v4/`.

같은 조합으로 새 탐색을 완료하면 그 폴더를 새 study의 best 결과로 갱신합니다. 다른 알고리즘·버전은 유지합니다. 탐색이 실패하면 이전에 저장된 best는 보존하고 이번 임시 파일을 정리합니다. 기존에 날짜별로 저장했던 과거 폴더는 이 코드에서 삭제하지 않습니다.

이 셀을 다시 실행하려면 **Optuna 준비 셀부터 다시 실행**하여 새 experiment를 만드세요. 현재 예시는 DB 탐색 재개나 pruning 없이 순차 실행합니다.

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.set_user_attr("model_name", model_name)
study.set_user_attr("model_version", model_version)
with experiment:
    study.optimize(objective, n_trials=n_trials)
    best_run_dir = experiment.save_best(study)

## 최적 결과 저장 및 확인

저장은 바로 앞 탐색 실행 셀에서 자동 완료됩니다. 이 셀은 `best_run_dir`의 결과를 확인합니다. 최적 모델은 이미 학습되어 있어 다시 학습할 필요가 없습니다.

한 폴더에 최적 모델 파일 한 개와 `best_parameters.yaml`을 저장합니다. 평가에 필요한 전처리·metadata·설정·validation 예측·지표·완료 상태도 함께 보관합니다. trial별 모델, 반복별 학습 이력, 반복되는 audit 파일은 영구 보관하지 않습니다.

- `trials.csv`: 이번 study의 모든 trial 파라미터·validation AP·상태만 기록
- `best_parameters.yaml`: 최적 모델의 전체 parameter 설정과 최적 모델 경로
- `summary.json`: 모델 이름·입력 버전·best AP·최적 모델 경로

예를 들어 XGBoost 버전 4를 여러 번 실행해도 `ML/models/best/xgboost_v4/` 한 곳을 갱신합니다. 탐색 완료 시 모델 YAML의 `model_path`와 버전별 `model_paths`를 자동 갱신하고, 기본 하이퍼파라미터 값은 유지합니다.

MLflow 전송 결과는 `summary.json`의 `mlflow`와 출력된 링크에서 확인합니다. 전송에 실패해도 로컬 best는 보존되며 `PENDING`으로 표시합니다. 아래 재시도 셀로 학습 없이 다시 전송할 수 있습니다. 원본 CSV·ECG 파형·환자별 예측 CSV는 이 자동 전송 대상이 아닙니다.

In [ ]:
summary = json.loads((best_run_dir / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("최적 모델과 파라미터 저장 폴더:", best_run_dir)
display(pd.read_csv(best_run_dir / "trials.csv"))
if summary.get("mlflow", {}).get("status") == "FINISHED":
    print("MLflow 결과:", summary["mlflow"]["url"])
else:
    print("MLflow 전송 상태:", summary.get("mlflow", {"status": "DISABLED"}))

## MLflow 전송 재시도 (필요할 때만)

서버 연결 문제로 `PENDING`이 표시되면 서버를 켠 뒤 이 셀만 실행합니다. 같은 best 결과는 기존 MLflow run으로 재시도하므로 중복 run을 만들지 않습니다. 새로운 탐색을 시작하기 전에 미전송 best를 먼저 전송하세요. 학습을 다시 할 필요는 없습니다.

In [ ]:
from Members.sang.model.tracking.log_best import log_best_to_mlflow

mlflow_result = log_best_to_mlflow(best_run_dir)
print("MLflow 결과:", mlflow_result["url"])

## 최종 test 평가

다른 후보와의 validation 비교까지 마친 뒤 모델·파라미터·threshold를 고정합니다. `evaluation/eval.yaml`에서 `target: ml`, `model_name`에 선택한 알고리즘 이름, `model_version`에 평가할 입력 버전(1~4), `artifact_dir: null`, `split: test`를 지정합니다. 평가 노트북 또는 평가 CLI는 해당 모델 `parameter.yaml`의 `model_path`를 읽어 저장 모델을 로드합니다. 평가 단계에서는 학습하지 않습니다.

특정 run을 직접 평가하려면 `artifact_dir`에 그 run 디렉터리를 지정합니다. 이 명시적 경로가 모델 YAML보다 우선합니다. `model_path`에 모델이 없으면 일반 학습이나 Optuna 탐색을 먼저 완료하라는 오류를 표시합니다.

AP는 threshold와 무관합니다. threshold를 조정하려면 validation 예측을 사용하고, test 결과를 보며 파라미터·threshold를 반복 조정하지 않습니다.